In [ ]:
import ccxt
import pandas as pd
import numpy as np
import time
from datetime import datetime 
import sys

exchange = ccxt.binance()
symbol = 'BTC/USDT'
timeframe = '5m'

# Remember last processed candle
last_candle_time = None

# --------------------------------
# 2. Fetch latest candles
# --------------------------------
def fetch_latest_data(limit=500):

    ohlcv = exchange.fetch_ohlcv(
        symbol=symbol,
        timeframe=timeframe,
        limit=limit
    )

    df = pd.DataFrame(
        ohlcv,
        columns=[
            'Timestamp',
            'Open',
            'High',
            'Low',
            'Close',
            'Volume'
        ]
    )

    df['Datetime'] = pd.to_datetime(
        df['Timestamp'],
        unit='ms'
    )

    df.set_index(
        'Datetime',
        inplace=True
    )
     
    return df

# --------------------------------
# 3. Session VWAP
# --------------------------------
def calculate_vwap(df):

    typical_price = (
        df['High']
        + df['Low']
        + df['Close']
    ) / 3

    tpv = typical_price * df['Volume']

    # Reset each UTC day
    date_group = df.index.date

    cumulative_tpv = tpv.groupby(date_group).cumsum()

    cumulative_volume = (df['Volume'].groupby(date_group).cumsum())

    df['VWAP'] = (cumulative_tpv / cumulative_volume)

    return df

# --------------------------------
# 4. ZScore
# --------------------------------
def calculate_zscore(df):

    deviation = (
        (df['Close'] - df['VWAP'])
        / df['VWAP']
    ) * 100

    rolling_mean = (deviation.rolling(50).mean())

    rolling_std = (deviation.rolling(50).std())

    df['ZScore'] = (deviation - rolling_mean) / rolling_std

    return df

# --------------------------------
# 5. Generate Signal
# --------------------------------
in_position = False
entry_price = None

def generate_signal(df):
    global in_position
    global entry_price

    latest = df.iloc[-2]
    prev = df.iloc[-3]

    z = latest['ZScore']

    entry_signal = (
        (prev['ZScore'] > -1)
        &
        (latest['ZScore'] <= -1)
    )

    exit_signal = (
        latest['ZScore'] >= 0
    )

    print("=" * 60)
    print(f"Candle Time: {latest.name}")

    print(
        f"Price: {latest['Close']:.2f}"
    )

    print(
        f"VWAP: {latest['VWAP']:.2f}"
    )

    print(
        f"ZScore: {z:.2f}"
    )

    print(
        f"Position: {'LONG' if in_position else 'FLAT'}"
    )
    
    # -------------------------
    # ENTRY
    # -------------------------
    if entry_signal and not in_position:

        in_position = True
        entry_price = latest['Close']

        print("🟢 BUY SIGNAL")
        print(
            f"Entered LONG @ "
            f"{entry_price:.2f}"
        )

    # -------------------------
    # EXIT
    # -------------------------
    elif exit_signal and in_position:

        exit_price = latest['Close']

        pnl_pct = (
            (exit_price - entry_price)
            / entry_price
        ) * 100

        print("🔴 EXIT SIGNAL")

        print(
            f"Exit Price: "
            f"{exit_price:.2f}"
        )

        print(
            f"Trade PnL: "
            f"{pnl_pct:.2f}%"
        )

        in_position = False
        entry_price = None

    else:
        print("⚪ NO ACTION")
    sys.stdout.flush()
    return df
# --------------------------------
# 6. Live Loop
# --------------------------------
while True:

    try:

        data = fetch_latest_data()

        data = calculate_vwap(data)

        data = calculate_zscore(data)
        
        latest_candle = data.index[-2]

        # Only run once per candle
        if latest_candle != last_candle_time:

            print("\nNEW CANDLE CLOSED", flush=True)

            generate_signal(data)

            last_candle_time = latest_candle

        else:
            print(
                f"{datetime.now()} Waiting for next candle...",
                flush=True
            )

    except Exception as e:

        print("Error:", e)

    # Wait 5 mins
    time.sleep(10)



NEW CANDLE CLOSED
Candle Time: 2026-05-17 08:10:00
Price: 78148.72
VWAP: 78036.54
ZScore: -0.31
Position: FLAT
⚪ NO ACTION
2026-05-17 04:18:11.813170 Waiting for next candle...
2026-05-17 04:18:22.016122 Waiting for next candle...
2026-05-17 04:18:32.210715 Waiting for next candle...
2026-05-17 04:18:42.398648 Waiting for next candle...
2026-05-17 04:18:52.581867 Waiting for next candle...
2026-05-17 04:19:02.762361 Waiting for next candle...
2026-05-17 04:19:13.005202 Waiting for next candle...
2026-05-17 04:19:23.192201 Waiting for next candle...
2026-05-17 04:19:33.369602 Waiting for next candle...
2026-05-17 04:19:43.558146 Waiting for next candle...
2026-05-17 04:19:53.748064 Waiting for next candle...

NEW CANDLE CLOSED
Candle Time: 2026-05-17 08:15:00
Price: 78085.41
VWAP: 78036.83
ZScore: -1.41
Position: FLAT
🟢 BUY SIGNAL
Entered LONG @ 78085.41
2026-05-17 04:20:14.115396 Waiting for next candle...
2026-05-17 04:20:24.333672 Waiting for next candle...
2026-05-17 04:20:34.55079